## Data Processing

In [1]:
import pyarrow.parquet as pq
import pyarrow as pa
import pandas as pd
import numpy as np

In [2]:
file_path = "..\\data\\Global Factor Data.parquet"
OUTPUT_FEATURES = "..\\data\\features_clean.parquet"
OUTPUT_TARGET = "..\\data\\target.parquet"

# Read full parquet file into Arrow Table
table = pq.read_table(file_path)
df = table.to_pandas()

display(df.head())

,obs_main,exch_main,common,primary_sec,gvkey,date,permno,permco,iid,id,...,dltnetis_mev,dstnetis_mev,dbnetis_mev,netis_mev,fincf_mev,ivol_capm_60m,beta_21d,beta_252d,rvol_252d,rvolhl_21d
0,1.0,1.0,1.0,1.0,040080,2024-12-31,22750.0,59043.0,02,22750.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.014801,0.003020,NaN
1,1.0,1.0,1.0,1.0,038617,2024-12-31,25327.0,59938.0,01,25327.0,...,0.000000,0.000000,0.000000,-0.000011,0.131983,NaN,2.325811,NaN,NaN,0.040160
2,1.0,1.0,1.0,1.0,144496,2024-12-31,15054.0,55100.0,02,15054.0,...,-0.058558,-0.619016,-0.677574,NaN,NaN,0.065632,1.058223,0.925554,0.015784,0.008192
3,1.0,1.0,1.0,1.0,037783,2024-12-31,20841.0,57760.0,02,20841.0,...,-0.376133,-0.368609,-0.744742,-0.092429,0.981088,0.337846,-0.025114,0.441205,0.093246,0.046725
4,1.0,1.0,1.0,1.0,038263,2024-12-31,25397.0,59969.0,01,25397.0,...,NaN,NaN,NaN,NaN,NaN,NaN,-0.352928,NaN,NaN,0.005995


In [3]:
cols_to_remove = {
    # identifiers and keys
    "permno", "permco", "eom",

    # administrative and data quality flags
    "obs_main", "exch_main", "primary_sec", "common", "size_grp",
    "ret_lag_dif", "adjfct", "bidask", "comp_tpci", "crsp_shrcd",
    "comp_exchg", "crsp_exchcd", "source_crsp", "curcd",

    # target variable
    "ret_exc_lead1m",

    # current period returns, for being risk of data leakage
    "ret_exc", "ret", "ret_local",

    # raw unscaled accounting numbers
    "enterprise_value", "book_equity", "assets", "sales", "net_income",

    # exchange rates and industry codes
    "fx", "gics", "naics", "sic", "ff49",

    # raw price and volume data
    "prc", "prc_local", "prc_high", "prc_low", "shares", "tvol",
}

# read schema only without data loading
schema = pq.read_schema(file_path)
all_columns = schema.names

print(f"Total columns in file: {len(all_columns)}")

# determine columns
existing_remove = cols_to_remove & set(all_columns)

feature_cols = [
    col for col in all_columns
    if col not in cols_to_remove and col != "ret_exc_lead1m"
]

target_col = ["ret_exc_lead1m"] if "ret_exc_lead1m" in all_columns else []

print(f"Columns being removed: {len(existing_remove)}")
print(f"Feature columns kept: {len(feature_cols)}")
print(f"Target column present: {bool(target_col)}")

# stream parquet in batches
parquet_file = pq.ParquetFile(file_path)

feature_writer = None
target_writer = None

for i, batch in enumerate(parquet_file.iter_batches(batch_size=500_000)):
    batch_table = pa.Table.from_batches([batch])

    # features
    feature_table = batch_table.select(feature_cols)

    if feature_writer is None:
        feature_writer = pq.ParquetWriter(
            OUTPUT_FEATURES,
            feature_table.schema,
            compression="snappy"
        )

    feature_writer.write_table(feature_table)

    # target
    if target_col:
        target_table = batch_table.select(target_col)

        if target_writer is None:
            target_writer = pq.ParquetWriter(
                OUTPUT_TARGET,
                target_table.schema,
                compression="snappy"
            )

        target_writer.write_table(target_table)

    print(f"Processed batch {i + 1} | rows: {batch_table.num_rows}")

# close writers
if feature_writer:
    feature_writer.close()

if target_writer:
    target_writer.close()

print(f"Features saved to: {OUTPUT_FEATURES}")
print(f"Target saved to: {OUTPUT_TARGET}")

# verify output
out_schema = pq.read_schema(OUTPUT_FEATURES)
print(f"Output feature columns: {len(out_schema.names)}")

Total columns in file: 443
Columns being removed: 37
Feature columns kept: 406
Target column present: True
Processed batch 1 | rows: 500000
Processed batch 2 | rows: 85595
Features saved to: ..\data\features_clean.parquet
Target saved to: ..\data\target.parquet
Output feature columns: 406


In [4]:
table_new = pq.read_table("..\\data\\features_clean.parquet")
df1 = table_new.to_pandas()
display(df1.head())

,gvkey,date,iid,id,excntry,me,me_company,dolvol,market_equity,div12m_me,...,dltnetis_mev,dstnetis_mev,dbnetis_mev,netis_mev,fincf_mev,ivol_capm_60m,beta_21d,beta_252d,rvol_252d,rvolhl_21d
0,040080,2024-12-31,02,22750.0,USA,50.46430,50.46430,8.829806e+08,50.46430,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.014801,0.003020,NaN
1,038617,2024-12-31,01,25327.0,USA,244.86336,244.86336,1.348946e+09,244.86336,NaN,...,0.000000,0.000000,0.000000,-0.000011,0.131983,NaN,2.325811,NaN,NaN,0.040160
2,144496,2024-12-31,02,15054.0,USA,97292.45328,97292.45328,8.219281e+10,97292.45328,0.028103,...,-0.058558,-0.619016,-0.677574,NaN,NaN,0.065632,1.058223,0.925554,0.015784,0.008192
3,037783,2024-12-31,02,20841.0,USA,11.59538,11.59538,1.185472e+09,11.59538,0.000000,...,-0.376133,-0.368609,-0.744742,-0.092429,0.981088,0.337846,-0.025114,0.441205,0.093246,0.046725
4,038263,2024-12-31,01,25397.0,USA,107.12000,107.12000,3.423243e+08,107.12000,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,-0.352928,NaN,NaN,0.005995


In [5]:
table_new2 = pq.read_table("..\\data\\target.parquet")
df2 = table_new2.to_pandas()
display(df2.head())

,ret_exc_lead1m
0,NaN
1,-0.499329
2,0.164909
3,0.151342
4,-0.010028


In [6]:
INPUT_FILE = "..\\data\\features_clean.parquet"

parquet_file = pq.ParquetFile(INPUT_FILE)

results = []

for i, batch in enumerate(parquet_file.iter_batches(batch_size=500_000)):

    batch_table = pa.Table.from_batches([batch])
    df_batch = batch_table.to_pandas()

    nan_counts = df_batch.isnull().sum(axis=1)
    results.append(nan_counts)

    print(f"Batch {i + 1} processed | rows: {len(df_batch)}")

nan_per_row = pd.concat(results, ignore_index=True)
print(f"Total rows: {len(nan_per_row)}")
print(f"Rows with zero NaNs: {(nan_per_row == 0).sum()}")
print(f"Rows with at least one NaN: {(nan_per_row > 0).sum()}")
print(f"Average NaNs per row: {nan_per_row.mean():.2f}")
print(f"Max NaNs in a single row: {nan_per_row.max()}")

Batch 1 processed | rows: 500000
Batch 2 processed | rows: 85595
Total rows: 585595
Rows with zero NaNs: 0
Rows with at least one NaN: 585595
Average NaNs per row: 95.33
Max NaNs in a single row: 400


## Missing Data processing


### Per-Stock Missing Data Filter

In [7]:
# Missing-data profile and row-level filter
INPUT_FILE = "..\\data\\features_clean.parquet"
FILTERED_OUTPUT = "..\\data\\features_missing_filtered.parquet"
MISSING_THRESHOLD = 1 / 3


def coerce_numeric_like_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Convert object columns to numeric only when every non-null value is numeric-like"""
    out = df.copy()

    for col in out.select_dtypes(include="object").columns:
        non_null = out[col].dropna()
        if non_null.empty:
            continue

        converted = pd.to_numeric(non_null, errors="coerce")
        if converted.notna().all():
            out[col] = pd.to_numeric(out[col], errors="coerce")

    return out


def detect_characteristic_columns(df: pd.DataFrame) -> list[str]:
    """Return continuous numeric feature columns using dtype/value checks, not name keywords"""
    char_cols = []

    for col in df.columns:
        series = df[col]

        if pd.api.types.is_datetime64_any_dtype(series):
            continue
        if not pd.api.types.is_numeric_dtype(series):
            continue

        values = series.dropna()
        if values.empty:
            continue

        n_unique = values.nunique(dropna=True)
        unique_ratio = n_unique / len(values)
        is_integer_like = np.isclose(values, values.round()).all()
        is_binary = n_unique <= 2
        is_low_cardinality = n_unique < 50
        is_identifier_like = is_integer_like and unique_ratio > 0.80

        # Keep continuous numeric characteristics. Exclude flags/categories and integer IDs.
        if is_binary or (is_integer_like and (is_low_cardinality or is_identifier_like)):
            continue

        char_cols.append(col)

    return char_cols


def add_missingness_columns(df: pd.DataFrame, char_cols: list[str]) -> pd.DataFrame:
    """Add per-row missing counts and missing fractions for characteristic columns"""
    out = df.copy()
    out["missing_count"] = out[char_cols].isna().sum(axis=1)
    out["missing_frac"] = out["missing_count"] / len(char_cols)
    return out


def summarise_missingness(df: pd.DataFrame, char_cols: list[str]) -> None:
    missing_by_char = df[char_cols].isna().mean().sort_values(ascending=False)

    print("Dataset")
    print(f"Rows: {len(df):,}")
    print(f"Columns: {df.shape[1]:,}")
    print(f"Stocks: {df['id'].nunique():,}" if "id" in df.columns else "Stocks: id column not found")
    print(f"Months: {df['date'].nunique():,}" if "date" in df.columns else "Months: date column not found")
    if "date" in df.columns:
        print(f"Date range : {df['date'].min().date()} to {df['date'].max().date()}")

    print("\nCharacteristic Columns")
    print(f"Detected: {len(char_cols):,}")
    print(f"First 10: {char_cols[:10]}")
    print(f"Last 10: {char_cols[-10:]}")

    print("\nMost Missing Characteristics")
    print(missing_by_char.head(10).map("{:.1%}".format).to_string())
    print(f"\nAverage characteristic missing rate: {missing_by_char.mean():.1%}")

    bins = [0, 0.10, 0.20, 1 / 3, 0.50, 0.75, 1.01]
    labels = ["0-10%", "10-20%", "20-33%", "33-50%", "50-75%", "75-100%"]
    missing_bins = pd.cut(df["missing_frac"], bins=bins, labels=labels, right=False)
    dist = missing_bins.value_counts().sort_index().rename("rows").to_frame()
    dist["pct"] = (dist["rows"] / len(df)).map("{:.1%}".format)
    dist["decision"] = np.where(dist.index.isin(["0-10%", "10-20%", "20-33%"]), "keep", "drop")

    print("\nPer-Row Missingness Distribution")
    print(dist.to_string())


def filter_missing_rows(df: pd.DataFrame, threshold: float = MISSING_THRESHOLD, missing_col: str = "missing_frac") -> pd.DataFrame:
    """Keep rows whose characteristic missing fraction is at or below threshold"""
    keep_mask = df[missing_col] <= threshold
    filtered = df.loc[keep_mask].copy()

    print("\nMissing-Data Filter")
    print(f"Threshold: {threshold:.1%}")
    print(f"Rows before: {len(df):,}")
    print(f"Rows kept: {len(filtered):,} ({len(filtered) / len(df):.1%})")
    print(f"Rows dropped: {(~keep_mask).sum():,} ({(~keep_mask).mean():.1%})")
    if "id" in filtered.columns:
        print(f"Stocks kept: {filtered['id'].nunique():,}")

    return filtered


# Load, profile, filter, and save.
df = pd.read_parquet(INPUT_FILE)
df = coerce_numeric_like_columns(df)
if "date" in df.columns:
    df["date"] = pd.to_datetime(df["date"])

char_cols = detect_characteristic_columns(df)
if not char_cols:
    raise ValueError("No characteristic columns were detected. Check the dtype-based detection rules.")

df_missing = add_missingness_columns(df, char_cols)
summarise_missingness(df_missing, char_cols)

df_filtered = filter_missing_rows(df_missing)
df_filtered.drop(columns=["missing_count", "missing_frac"], errors="ignore").to_parquet(
    FILTERED_OUTPUT, index=False,
)

print(f"\nFiltered dataset saved to: {FILTERED_OUTPUT}")
print(f"Filtered shape: {df_filtered.drop(columns=['missing_count', 'missing_frac'], errors='ignore').shape}")

Dataset
Rows: 585,595
Columns: 408
Stocks: 48,763
Months: 308
Date range : 2024-12-31 to 2025-12-31

Characteristic Columns
Detected: 399
First 10: ['gvkey', 'id', 'me', 'me_company', 'dolvol', 'market_equity', 'div12m_me', 'chcsho_12m', 'eqnpo_12m', 'ret_1_0']
Last 10: ['dltnetis_mev', 'dstnetis_mev', 'dbnetis_mev', 'netis_mev', 'fincf_mev', 'ivol_capm_60m', 'beta_21d', 'beta_252d', 'rvol_252d', 'rvolhl_21d']

Most Missing Characteristics
pstk_gr3      97.4%
pstk_gr1      97.3%
adv_sale      95.7%
eqpo_gr3a     83.9%
div1m_me      83.9%
eqbb_gr3a     83.0%
eqpo_gr1a     80.2%
staff_sale    79.9%
eqis_gr3a     79.1%
eqbb_gr1a     79.0%

Average characteristic missing rate: 23.4%

Per-Row Missingness Distribution
                rows    pct decision
missing_frac                        
0-10%         190659  32.6%     keep
10-20%        192080  32.8%     keep
20-33%         63138  10.8%     keep
33-50%         70495  12.0%     drop
50-75%         25967   4.4%     drop
75-100%        4325

### Rank Standardisation and NaN Replacement

For each month (cross-section):
  1. Rank each characteristic cross-sectionally using pct=True to [0, 1]
  2. NaNs are assigned rank 0.5 (the midpoint) via na_option='keep' + fillna
  3. Subtract 0.5 to centre the distribution to [-0.5, 0.5]
  4. NaNs become 0.0 (the cross-sectional median)

This follows the protocol mentioned in Kelly, Kuznetsov, Malamud & Xu (2025).

In [8]:
# Cross-sectional rank standardisation with full NaN cleanup
RANK_INPUT = globals().get("FILTERED_OUTPUT", "..\\data\\features_missing_filtered.parquet")
RANKED_OUTPUT = "..\\data\\features_ranked.parquet"


def rank_standardise_by_date(df: pd.DataFrame, char_cols: list[str], date_col: str = "date") -> pd.DataFrame:
    """Rank-standardise characteristics within each date cross-section."""
    if date_col not in df.columns:
        raise KeyError(f"'{date_col}' is required for cross-sectional standardisation.")

    ranked = df.copy()
    ranks = ranked.groupby(date_col, group_keys=False)[char_cols].rank(
        method="average",
        pct=True,
        na_option="keep",
    )
    ranked[char_cols] = ranks.fillna(0.5) - 0.5
    return ranked


def fill_remaining_non_characteristic_nans(df: pd.DataFrame, char_cols: list[str], date_col: str = "date") -> pd.DataFrame:
    """Fill NaNs left in identifier, size, and metadata columns after characteristic ranking"""
    out = df.copy()
    non_char_cols = [col for col in out.columns if col not in char_cols]
    remaining_na = out[non_char_cols].isna().sum()
    remaining_na = remaining_na[remaining_na > 0].sort_values(ascending=False)

    if remaining_na.empty:
        print("No NaNs remain in non-characteristic columns")
        return out

    print("\nRemaining NaNs Outside Characteristic Columns")
    print(remaining_na.to_string())

    numeric_cols = out[remaining_na.index].select_dtypes(include="number").columns.tolist()
    object_cols = out[remaining_na.index].select_dtypes(include="object").columns.tolist()
    text_cols = []

    for col in object_cols:
        non_null = out[col].dropna()
        converted = pd.to_numeric(non_null, errors="coerce")
        if not non_null.empty and converted.notna().all():
            out[col] = pd.to_numeric(out[col], errors="coerce")
            numeric_cols.append(col)
        else:
            text_cols.append(col)

    for col in numeric_cols:
        if date_col in out.columns:
            monthly_median = out.groupby(date_col)[col].transform("median")
            out[col] = out[col].fillna(monthly_median)
        global_median = out[col].median()
        fallback = 0.0 if pd.isna(global_median) else global_median
        out[col] = out[col].fillna(fallback)

    for col in text_cols:
        mode = out[col].mode(dropna=True)
        fallback = "missing" if mode.empty else mode.iloc[0]
        out[col] = out[col].fillna(fallback).astype("string")

    datetime_cols = out[remaining_na.index].select_dtypes(include="datetimetz").columns.tolist()
    datetime_cols += out[remaining_na.index].select_dtypes(include="datetime").columns.tolist()
    for col in datetime_cols:
        mode = out[col].mode(dropna=True)
        if not mode.empty:
            out[col] = out[col].fillna(mode.iloc[0])

    return out


def verify_ranked_data( original: pd.DataFrame, ranked: pd.DataFrame, char_cols: list[str], date_col: str = "date") -> None:
    values = ranked[char_cols]
    n_missing_chars = int(values.isna().sum().sum())
    n_missing_total = int(ranked.isna().sum().sum())
    min_value = values.min().min()
    max_value = values.max().max()

    print("\n Ranked Data Checks")
    print(f"NaNs remaining in characteristics: {n_missing_chars:,}")
    print(f"NaNs remaining in full dataset: {n_missing_total:,}")
    print(f"Characteristic range: [{min_value:.4f}, {max_value:.4f}]")

    assert n_missing_chars == 0, "NaNs remain in characteristic columns."
    assert n_missing_total == 0, "NaNs remain somewhere in the full ranked dataset."
    assert min_value >= -0.5 - 1e-9, "Values below -0.5 detected."
    assert max_value <= 0.5 + 1e-9, "Values above 0.5 detected."

    example_col = char_cols[0]
    original_nan_mask = original[example_col].isna()
    if original_nan_mask.any():
        imputed_values = ranked.loc[original_nan_mask, example_col]
        all_zero = np.isclose(imputed_values, 0.0).all()
        print(f"Original NaNs in '{example_col}' mapped to 0.0: {all_zero}")
        assert all_zero, "Original characteristic NaNs were not mapped to 0.0."

    monthly_means = ranked.groupby(date_col)[char_cols].mean().mean(axis=1)
    print(f"Average monthly cross-sectional mean: {monthly_means.mean():.6f}")
    print(f"Min monthly mean: {monthly_means.min():.6f}")
    print(f"Max monthly mean: {monthly_means.max():.6f}")


# Load the filtered data and reuse the same characteristic-detection rule.
df_filtered = pd.read_parquet(RANK_INPUT)
df_filtered = coerce_numeric_like_columns(df_filtered)
df_filtered["date"] = pd.to_datetime(df_filtered["date"])

char_cols = detect_characteristic_columns(df_filtered)
if not char_cols:
    raise ValueError("No characteristic columns were detected. Check the exclusion rules.")

print(f"Rows to standardise: {len(df_filtered):,}")
print(f"Characteristic columns: {len(char_cols):,}")

print("\nApplying rank standardisation by month")
df_ranked = rank_standardise_by_date(df_filtered, char_cols)
df_ranked = fill_remaining_non_characteristic_nans(df_ranked, char_cols)
verify_ranked_data(df_filtered, df_ranked, char_cols)

df_ranked.to_parquet(RANKED_OUTPUT, index=False)
print(f"\nRanked dataset saved to: {RANKED_OUTPUT}")
print(f"Ranked shape: {df_ranked.shape}")

Rows to standardise: 446,565
Characteristic columns: 399

Applying rank standardisation by month

Remaining NaNs Outside Characteristic Columns
divspc1m_me     421161
divspc12m_me    239921
ni_inc8q         85436
f_score          31779

 Ranked Data Checks
NaNs remaining in characteristics: 0
NaNs remaining in full dataset: 0
Characteristic range: [-0.5000, 0.5000]
Average monthly cross-sectional mean: 0.054123
Min monthly mean: 0.000015
Max monthly mean: 0.414787

Ranked dataset saved to: ..\data\features_ranked.parquet
Ranked shape: (446565, 406)


In [9]:
INPUT_FILE = "..\\data\\features_ranked.parquet"

parquet_file = pq.ParquetFile(INPUT_FILE)

results = []

for i, batch in enumerate(parquet_file.iter_batches(batch_size=500_000)):

    batch_table = pa.Table.from_batches([batch])
    df_batch = batch_table.to_pandas()

    nan_counts = df_batch.isnull().sum(axis=1)
    results.append(nan_counts)

    print(f"Batch {i + 1} processed | rows: {len(df_batch)}")

nan_per_row = pd.concat(results, ignore_index=True)

print(f"Total rows: {len(nan_per_row)}")
print(f"Rows with zero NaNs: {(nan_per_row == 0).sum()}")
print(f"Rows with at least one NaN: {(nan_per_row > 0).sum()}")
print(f"Average NaNs per row: {nan_per_row.mean():.2f}")
print(f"Max NaNs in a single row: {nan_per_row.max()}")

Batch 1 processed | rows: 446565
Total rows: 446565
Rows with zero NaNs: 446565
Rows with at least one NaN: 0
Average NaNs per row: 0.00
Max NaNs in a single row: 0
